## Домашнее задание 2. Решающие деревья

### О задании

Задание было адаптировано из курса по машинному обучению Е.А. Соколова.

### Оценивание и штрафы
Каждая из частей задания имеет определенную «стоимость» (указана в скобках около названия части). Максимально допустимая оценка за работу — 5 баллов. Помимо концептуальных ошибок и отсутствия решения, оценка может быть снижена за неэффективную реализацию или неопрятные графики.

**Обратите внимание**, что в каждом разделе домашнего задания есть оцениваемые задачи и есть вопросы. Вопросы дополняют задачи и направлены на то, чтобы проинтерпретировать или обосновать происходящее. Код без интерпретации не имеет смысла, поэтому отвечать на вопросы обязательно — за отсутствие ответов баллы будут снижены. Если вы ответите на вопросы, но не напишете корректный код к соответствующим оцениваемым задачам, то баллы за такое выставлены не будут.

### Формат сдачи
Задания сдаются через ваш приватный гит репозиторий. Туда необходимо сложить ноутбук с вашим решением и модуль `hw2code.py` с реализованными функциями и классами.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

%matplotlib inline

sns.set(style="whitegrid")

import warnings
warnings.filterwarnings('ignore')

## Часть 1. Решающие деревья: визуализация (1.5 балла)

В этой части рассмотрим три простых двумерных датасета, сделанных с помощью `make_circles`, `make_moons` и `make_classification`, и посмотрим, как ведет себя разделяющая поверхность в зависимости от различных гиперпараметров.

In [ ]:
from sklearn.datasets import make_moons, make_circles, make_classification

datasets = [
    make_circles(noise=0.2, factor=0.5, random_state=42),
    make_moons(noise=0.2, random_state=42),
    make_classification(n_classes=3, n_clusters_per_class=1, n_features=2, class_sep=.8, random_state=3,
                        n_redundant=0),
]

In [ ]:
palette = sns.color_palette(n_colors=3)
cmap = ListedColormap(palette)

In [ ]:
plt.figure(figsize=(15, 4))
for i, (x, y) in enumerate(datasets):
    plt.subplot(1, 3, i + 1)
    plt.scatter(x[:, 0], x[:, 1], c=y, cmap=cmap, alpha=.8)

**Задание 1**. Для каждого датасета обучите решающее дерево с параметрами по умолчанию, предварительно разбив выборку на обучающую и тестовую. Постройте разделяющие поверхности (для этого воспользуйтесь функцией `plot_surface`, пример ниже). Посчитайте accuracy на обучающей и тестовой выборках.

In [ ]:
def plot_surface(clf, X, y):
    plot_step = 0.01
    palette = sns.color_palette(n_colors=len(np.unique(y)))
    cmap = ListedColormap(palette)
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, plot_step),
                         np.arange(y_min, y_max, plot_step))
    plt.tight_layout(h_pad=0.5, w_pad=0.5, pad=2.5)

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    cs = plt.contourf(xx, yy, Z, cmap=cmap, alpha=0.3)

    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap, alpha=.7,
                edgecolors=np.array(palette)[y], linewidths=2)

In [ ]:
# Пример:
from sklearn.linear_model import LinearRegression

X, y = datasets[2]
lr = LinearRegression().fit(X, y)
plot_surface(lr, X, y)

**Вопрос**: Сильно ли деревья переобучились?

**Задание 2**. Попробуйте перебрать несколько параметров для регуляризации (напр. `max_depth`, `min_samples_leaf`). Для каждого набора гиперпараметров постройте разделяющую поверхность, выведите обучающую и тестовую ошибки. Можно делать кросс-валидацию или просто разбиение на трейн и тест, главное делайте каждый раз одинаковое разбиение, чтобы можно было корректно сравнивать (помним, что дерево сильно зависит от небольшого изменения обучающей выборки). Проследите, как меняются разделяющая поверхность и обобщающая способность.

**Вопрос**: Почему разделяющая поверхность и обобщающая способность меняются именно так? Одинаково ли это изменение для разных датасетов?

## Часть 2. Решающие деревья своими руками (3.5 балла)

В этой части вам нужно реализовать свой класс для обучения решающего дерева в задаче бинарной классификации с возможностью обработки вещественных и категориальных признаков.

**Задание 3**. Реализуйте функцию `find_best_split` из модуля `hw2code.py`. Немного подробнее внутри докстринга самого файла.

**Задание 4**. Загрузите таблицу `students.csv` (она лежит рядом с ноутбуком; это немного преобразованный датасет [User Knowledge](https://archive.ics.uci.edu/ml/datasets/User+Knowledge+Modeling)). В ней признаки объекта записаны в первых пяти столбцах, а в последнем записана целевая переменная (класс: 0 или 1). Постройте на одном изображении пять кривых «порог — значение критерия Джини» для всех пяти признаков. Отдельно визуализируйте scatter-графики «значение признака — класс» для всех пяти признаков.

**Вопрос**: Исходя из кривых значений критерия Джини, по какому признаку нужно производить деление выборки на два поддерева? Согласуется ли этот результат с визуальной оценкой scatter-графиков? Как бы вы охарактеризовали вид кривой для «хороших» признаков, по которым выборка делится почти идеально? Чем отличаются кривые для признаков, по которым деление практически невозможно?

**Задание 5**. Разберитесь с уже написанным кодом в классе `DecisionTree` модуля `hw2code.py`. Найдите ошибки в реализации метода `_fit_node`. Напишите функцию `_predict_node`.

**Список найденных ошибок**:

1. ...
2. ...

Построение дерева осуществляется согласно базовому жадному алгоритму, предложенному в [лекции](https://github.com/esokolov/ml-course-hse/blob/master/2020-fall/lecture-notes/lecture07-trees.pdf) Е.А. Соколова в разделе «Построение дерева». Выбор лучшего разбиения необходимо производить по критерию Джини. Критерий останова: все объекты в листе относятся к одному классу или ни по одному признаку нельзя разбить выборку. Ответ в листе: наиболее часто встречающийся класс в листе. Для категориальных признаков выполняется преобразование, описанное в лекции в разделе «Учет категориальных признаков».

**Задание 6**. Протестируйте свое решающее дерево на датасете [mushrooms](https://archive.ics.uci.edu/ml/datasets/Mushroom). Вам нужно скачать таблицу agaricus-lepiota.data, прочитать ее с помощью `pandas`, применить к каждому столбцу `LabelEncoder` (из `sklearn`), чтобы преобразовать строковые имена категорий в натуральные числа. Первый столбец — это целевая переменная (e — edible, p — poisonous). Мы будем измерять качество с помощью accuracy, так что нам не очень важно, что будет классом 1, а что — классом 0. Обучите решающее дерево на половине случайно выбранных объектов (признаки в датасете категориальные) и сделайте предсказания для оставшейся половины. Вычислите accuracy.

У вас должно получиться значение accuracy, равное единице (или очень близкое к единице), и не очень глубокое дерево.

**Задание 7**. Загрузите следующие наборы данных (`pandas` умеет загружать файлы по url, в нашем случае это файл \*.data), предварительно ознакомившись с описанием признаков и целевой переменной в каждом из них (она записаны в Data Folder, в файле \*.names):
* [mushrooms](https://archive.ics.uci.edu/ml/datasets/Mushroom) (загрузили в предыдущем пункте, классы записаны в нулевом столбце),
* [cars](https://archive.ics.uci.edu/ml/datasets/Car+Evaluation) (классы записаны в последнем столбце, считаем, что unacc, acc — это класс 0, good, vgood — класс 1),
* [nursery](https://archive.ics.uci.edu/ml/datasets/Nursery) (классы записаны в последнем столбце, считаем, что not_recom и recommend — класс 0, very_recom, priority, spec_prior — класс 1).

Закодируйте категориальные признаки, использовав `LabelEncoder`. С помощью `cross_val_score` (`cv=10`) оцените accuracy на каждом из этих наборов данных следующих алгоритмов:
* `DecisionTree`, считающий все признаки вещественными,
* `DecisionTree`, считающий все признаки категориальными,
* `DecisionTree`, считающий все признаки вещественными + one-hot-encoding всех признаков,
* `DecisionTreeClassifier` из `sklearn`.

Запишите результат в `pd.DataFrame` (по строкам — наборы данных, по столбцам — алгоритмы). Также было бы здорово, если представите компанктую и понятную визуализацию этих результатов.

Рекомендации:
* Чтобы `cross_val_score` вычисляла точность, нужно передать `scoring=make_scorer(accuracy_score)`, обе функции из `sklearn.metrics`.
* Если вам позволяет память (а она скорее всего позволяет), указывайте параметр `sparse_output=False` в `OneHotEncoder` (если вы, конечно, используете его). Иначе вам придется добиваться того, чтобы ваша реализация дерева умела работать с разреженными матрицами (что тоже, в целом, не очень сложно).

**Вопрос**: Проанализируйте результаты эксперимента.
Одинаково ли для разных наборов данных ранжируются алгоритмы?
Порассуждайте, почему так происходит.

## Часть N.

Вставьте картинку, описывающую ваш опыт выполнения этого ДЗ.